# Local Network Geometry

This notebook is dedicated to comparing hidden-state manifold geometry for `Local-Hom` and `Local-Het` on SHD.

It implements the main geometry analyses motivated by the three manifold papers:
1. Windowed hidden-state PCA visualizations.
2. Time-resolved geometry metrics: participation ratio, class separation, factorization, and noise alignment.
3. Few-shot linear probe curves.
4. Cross-time decoding heatmaps.
5. Unit-by-window variance matrices with clustering.

`Local-Het` is loaded from the saved checkpoint. `Local-Hom` is loaded from its own checkpoint if available; otherwise this notebook can optionally train and save it.

In [1]:
from __future__ import annotations

import sys
import json
import random
import time
from contextlib import contextmanager
from pathlib import Path

import importlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path(r"C:\Users\Priya\Desktop\research project (SNN Info Theory)")
WIMFO_ROOT = PROJECT_ROOT / "wimfo"
PAPER_ROOT = PROJECT_ROOT / "neural_heterogeneity" / "SuGD_code"
for extra_path in [WIMFO_ROOT, PAPER_ROOT]:
    if str(extra_path) not in sys.path:
        sys.path.insert(0, str(extra_path))

from clipper import ZeroOneClipper
from data_gen import open_file, sparse_data_generator
from reg_loss import loss as repo_loss
from SuSpike import SuSpike

RSNN = importlib.import_module("model").RSNN
clipper = ZeroOneClipper()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
spike_fn = SuSpike.apply

SHD_TRAIN = PROJECT_ROOT / "data" / "shd" / "shd_train.h5"
SHD_TEST = PROJECT_ROOT / "data" / "shd" / "shd_test.h5"
LOCAL_HET_CHECKPOINT_PATH = PROJECT_ROOT / "Project Files" / "network_A_checkpoint.pt"
LOCAL_HOM_CHECKPOINT_PATH = PROJECT_ROOT / "Project Files" / "local_hom_checkpoint.pt"
GEOMETRY_OUTPUT_DIR = PROJECT_ROOT / "Project Files" / "geometry_outputs"

print(f"Device: {DEVICE}")
print(f"Local-Het checkpoint exists: {LOCAL_HET_CHECKPOINT_PATH.exists()}")
print(f"Local-Hom checkpoint exists: {LOCAL_HOM_CHECKPOINT_PATH.exists()}")


Device: cuda
Local-Het checkpoint exists: True
Local-Hom checkpoint exists: False


In [2]:
# Shared model, data, and training helpers.

SHD_DEFAULTS = {
    "seed": 1000,
    "dtype": torch.float,
    "device": DEVICE,
    "cuda": DEVICE.type == "cuda",
    "nb_inputs": 700,
    "nb_hidden": [],
    "nb_recurrent": 32,
    "nb_outputs": 20,
    "batch_size": 256,
    "time_step": 1.0e-3,
    "nb_steps": 1000,
    "tau_syn": 10e-3,
    "tau_mem": 20e-3,
    "threshold": 1.0,
    "tref": 0.0,
    "dist": "gamma",
    "dist_prms": 3.0,
    "lr": 4e-3,
    "lr_ab": 4e-3,
    "betas": (0.9, 0.999),
    "weight_decay": 0.0,
    "nb_epochs": 25,
    "drop_last": True,
    "sl": 0.0,
    "thetal": 0.0,
    "su": 0.0,
    "thetau": 0.0,
    "rate": 0.0,
    "p_del": 0.0,
    "train_th": 0,
    "het_th": 0,
    "train_reset": 0,
    "het_reset": 0,
    "train_rest": 0,
    "het_rest": 0,
    "sparse_data_generator": "sparse_data_generator",
    "time_scale": [0.5, 1.0],
    "model": "RSNN",
    "savestep": 10,
    "clip": 1,
    "plot_step": 50,
    "class_list": list(range(20)),
    "het_ab": 1,
    "train_ab": 1,
    "train_hom_ab": 0,
}

GEOM_CONFIG = {
    "auto_train_local_hom_if_missing": False,
    "save_local_hom_after_train": True,
    "max_test_batches": None,
    "representation": "mem_mean",
    "n_windows": 5,
    "fewshot_shots": [1, 2, 4, 8, 16, 32],
    "fewshot_repeats": 6,
    "cross_time_train_shots": 16,
    "cross_time_repeats": 3,
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_optimizer(model, prms):
    weight_params, ab_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "alpha" in name or "beta" in name:
            ab_params.append(param)
        else:
            weight_params.append(param)
    param_groups = [{"params": weight_params, "lr": prms["lr"], "weight_decay": prms["weight_decay"]}]
    if ab_params:
        param_groups.append({"params": ab_params, "lr": prms["lr_ab"]})
    return torch.optim.Adam(param_groups, betas=prms["betas"])


@contextmanager
def shd_open(path):
    units, times, labels = open_file(str(path))
    try:
        yield units, times, labels
    finally:
        units._v_file.close()


class SHDCache:
    def __init__(self, path):
        raw_u, raw_t, raw_l = open_file(str(path))
        self.units = list(raw_u[:])
        self.times = list(raw_t[:])
        self.labels = np.array(raw_l[:])
        raw_u._v_file.close()
        print(f"  SHDCache: {len(self.labels)} samples loaded from {Path(str(path)).name}")

    def __len__(self):
        return len(self.labels)


def _is_cache(obj):
    return hasattr(obj, "units") and hasattr(obj, "times") and hasattr(obj, "labels")


@contextmanager
def shd_open_cached(cache):
    yield cache.units, cache.times, cache.labels


def fast_sparse_data_generator(units, times, labels, prms, shuffle=True, epoch=0, drop_last=True):
    rate = prms.get("rate", 0.0)
    p_del = prms.get("p_del", 0.0)
    if rate != 0.0 or p_del != 0.0:
        yield from sparse_data_generator(units, times, labels, prms, shuffle=shuffle, epoch=epoch, drop_last=drop_last)
        return

    seed = prms["seed"] + epoch
    batch_size = prms["batch_size"]
    nb_steps = prms["nb_steps"]
    nb_units = prms["nb_inputs"]
    inv_dt = 1.0 / prms["time_step"]
    class_list = prms["class_list"]

    label_arr = labels if isinstance(labels, np.ndarray) else np.array(labels[:])
    sample_index = np.where(np.isin(label_arr, class_list))[0]
    num_samples = len(sample_index)
    n_batches = (num_samples // batch_size) if drop_last else -(-num_samples // batch_size)

    np.random.seed(seed)
    if shuffle:
        np.random.shuffle(sample_index)

    for counter in range(n_batches):
        batch_index = sample_index[batch_size * counter:min(num_samples, batch_size * (counter + 1))]
        actual_bs = len(batch_index)

        t_arrays = [np.round(times[idx] * inv_dt).astype(np.int64) for idx in batch_index]
        u_arrays = [units[idx] for idx in batch_index]
        lengths = np.array([len(a) for a in t_arrays], dtype=np.int64)

        if lengths.sum():
            all_ts = np.concatenate(t_arrays)
            all_us = np.concatenate(u_arrays)
            all_bc = np.repeat(np.arange(actual_bs, dtype=np.int64), lengths)
            valid = all_ts < nb_steps
            all_ts, all_us, all_bc = all_ts[valid], all_us[valid], all_bc[valid]
            i = torch.from_numpy(np.stack([all_bc, all_ts, all_us]))
            v = torch.ones(all_ts.size, dtype=torch.float32)
            X_batch = torch.sparse_coo_tensor(i, v, torch.Size([actual_bs, nb_steps, nb_units])).to_dense()
        else:
            X_batch = torch.zeros(actual_bs, nb_steps, nb_units)

        X_batch.clamp_(max=1.0)
        y_batch = torch.tensor([class_list.index(int(a)) for a in label_arr[batch_index]], dtype=torch.long)
        yield X_batch, y_batch


def shd_generator(units, times, labels, prms, shuffle, epoch, drop_last):
    yield from fast_sparse_data_generator(units, times, labels, prms, shuffle=shuffle, epoch=epoch, drop_last=drop_last)


def forward_logits(model, x):
    layer_recs = model(0, 0, x)
    output_layer = layer_recs[-1]
    logits, _ = torch.max(output_layer[1], dim=1)
    return logits, layer_recs


@torch.no_grad()
def evaluate_batches(model, prms, units, times, labels):
    total = int(np.isin(labels[:], prms["class_list"]).sum())
    model.eval()
    correct = 0
    for x, y in shd_generator(units, times, labels, prms, shuffle=False, epoch=0, drop_last=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits, _ = forward_logits(model, x)
        correct += (logits.argmax(1) == y).sum().item()
    return {"acc": correct / max(total, 1), "n": total}


def train_experiment(model, prms, train_data, test_data, use_amp=True):
    use_amp = use_amp and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    optimizer = make_optimizer(model, prms)
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    tr_ctx = shd_open_cached if _is_cache(train_data) else shd_open
    te_ctx = shd_open_cached if _is_cache(test_data) else shd_open
    with tr_ctx(train_data) as (u_tr, t_tr, l_tr), te_ctx(test_data) as (u_te, t_te, l_te):
        total_tr = int(np.isin(l_tr[:], prms["class_list"]).sum())
        total_te = int(np.isin(l_te[:], prms["class_list"]).sum())

        if prms["clip"]:
            model.apply(clipper)

        for epoch in range(1, prms["nb_epochs"] + 1):
            epoch_correct = 0
            epoch_loss = 0.0
            model.train()
            for x, y in shd_generator(u_tr, t_tr, l_tr, prms, shuffle=True, epoch=epoch, drop_last=prms["drop_last"]):
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                    logits, layer_recs = forward_logits(model, x)
                    loss_val = repo_loss(logits, layer_recs, y, total_tr, prms)
                scaler.scale(loss_val).backward()
                scaler.step(optimizer)
                scaler.update()
                if prms["clip"]:
                    model.apply(clipper)
                epoch_loss += loss_val.item()
                epoch_correct += (logits.argmax(1) == y).sum().item()

            test_metrics = evaluate_batches(model, prms, u_te, t_te, l_te)
            history["train_loss"].append(epoch_loss)
            history["train_acc"].append(epoch_correct / max(total_tr, 1))
            history["test_loss"].append(float("nan"))
            history["test_acc"].append(test_metrics["acc"])
            print(f"epoch={epoch:03d} train_acc={history['train_acc'][-1]:.3f} test_acc={test_metrics['acc']:.3f}")
    return history


print("Shared helpers loaded.")

Shared helpers loaded.


In [3]:
print("Pre-loading SHD training data into RAM...")
SHD_TRAIN_CACHE = SHDCache(SHD_TRAIN)
print("Pre-loading SHD test data into RAM...")
SHD_TEST_CACHE = SHDCache(SHD_TEST)


def build_prms_from_checkpoint_prms(saved_prms, heterogeneous):
    prms = dict(SHD_DEFAULTS)
    prms.update(saved_prms)
    prms.update({
        "dtype": torch.float,
        "device": DEVICE,
        "cuda": DEVICE.type == "cuda",
        "het_ab": int(heterogeneous),
        "train_ab": int(heterogeneous),
        "train_hom_ab": int(not heterogeneous),
    })
    prms["alpha"] = float(np.exp(-prms["time_step"] / prms["tau_syn"]))
    prms["beta"] = float(np.exp(-prms["time_step"] / prms["tau_mem"]))
    return prms


local_het_ckpt = torch.load(LOCAL_HET_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
local_het_prms = build_prms_from_checkpoint_prms(dict(local_het_ckpt["prms"]), heterogeneous=True)
LOCAL_HET_LABEL = "Local-Het"
LOCAL_HOM_LABEL = "Local-Hom"

set_seed(local_het_prms["seed"])
local_het_model = RSNN(local_het_prms, rec=True).to(DEVICE)
local_het_model.load_state_dict(local_het_ckpt["model_state_dict"])
local_het_history = local_het_ckpt["history"]
local_het_elapsed = float(local_het_ckpt["elapsed_s"])

local_hom_prms = build_prms_from_checkpoint_prms(dict(local_het_ckpt["prms"]), heterogeneous=False)
local_hom_model = None
local_hom_history = None
local_hom_elapsed = float("nan")

if LOCAL_HOM_CHECKPOINT_PATH.exists():
    local_hom_ckpt = torch.load(LOCAL_HOM_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    set_seed(local_hom_prms["seed"])
    local_hom_model = RSNN(local_hom_prms, rec=True).to(DEVICE)
    local_hom_model.load_state_dict(local_hom_ckpt["model_state_dict"])
    local_hom_history = local_hom_ckpt.get("history")
    local_hom_elapsed = float(local_hom_ckpt.get("elapsed_s", float("nan")))
    print(f"Loaded {LOCAL_HOM_LABEL} checkpoint.")
elif GEOM_CONFIG["auto_train_local_hom_if_missing"]:
    print(f"No {LOCAL_HOM_LABEL} checkpoint found at {LOCAL_HOM_CHECKPOINT_PATH.name}. Training now...")
    set_seed(local_hom_prms["seed"])
    local_hom_model = RSNN(local_hom_prms, rec=True).to(DEVICE)
    t0 = time.perf_counter()
    local_hom_history = train_experiment(local_hom_model, local_hom_prms, SHD_TRAIN_CACHE, SHD_TEST_CACHE, use_amp=True)
    local_hom_elapsed = time.perf_counter() - t0
    if GEOM_CONFIG["save_local_hom_after_train"]:
        torch.save({
            "model_state_dict": local_hom_model.state_dict(),
            "history": local_hom_history,
            "elapsed_s": local_hom_elapsed,
            "prms": {k: v for k, v in local_hom_prms.items() if not isinstance(v, (torch.dtype, torch.device))},
        }, LOCAL_HOM_CHECKPOINT_PATH)
        print(f"Saved {LOCAL_HOM_LABEL} checkpoint -> {LOCAL_HOM_CHECKPOINT_PATH.name}")
else:
    print(
        f"{LOCAL_HOM_LABEL} checkpoint missing. To continue, either save one to {LOCAL_HOM_CHECKPOINT_PATH.name} "
        "or set GEOM_CONFIG['auto_train_local_hom_if_missing'] = True and rerun this cell."
    )

local_het_eval = evaluate_batches(local_het_model, local_het_prms, SHD_TEST_CACHE.units, SHD_TEST_CACHE.times, SHD_TEST_CACHE.labels)
print(f"{LOCAL_HET_LABEL}: test_acc={local_het_eval['acc']:.3f}")
if local_hom_model is not None:
    local_hom_eval = evaluate_batches(local_hom_model, local_hom_prms, SHD_TEST_CACHE.units, SHD_TEST_CACHE.times, SHD_TEST_CACHE.labels)
    print(f"{LOCAL_HOM_LABEL}: test_acc={local_hom_eval['acc']:.3f}")


Pre-loading SHD training data into RAM...
  SHDCache: 8156 samples loaded from shd_train.h5
Pre-loading SHD test data into RAM...
  SHDCache: 2264 samples loaded from shd_test.h5
Local-Hom checkpoint missing. To continue, either save one to local_hom_checkpoint.pt or set GEOM_CONFIG['auto_train_local_hom_if_missing'] = True and rerun this cell.
Local-Het: test_acc=0.649
